In [1]:
import argparse
import xgboost as xgb
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score
import h5py
import torch

In [2]:
# read resources
Xreducedall = np.load('/media/ggj/Guo-4T-AI/FYT/MultiOmic/CodeTest/20240819_epigen_pred_expr/metabrain_sc/model_0825/Xreducedall_all.npy')
geneanno = pd.read_csv('/media/ggj/Guo-4T-AI/FYT/MultiOmic/CodeTest/20240819_epigen_pred_expr/geneanno.csv')

geneexp = pd.read_csv('exp.csv',index_col=0)
geneexp

,AST-PP,IN-PV,IN-SST,IN-SV2C,IN-VIP,L2/3,L4,L5/6,L5/6-CC,Microglia,OPC,Oligodendrocytes
DDX11L1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
WASH7P,0.123136,0.156127,0.173884,0.221039,0.170023,0.186092,0.240504,0.192655,0.226548,0.078538,0.060653,0.042938
MIR6859-3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
RP11-34P13.3,0.000000,0.002255,0.000029,0.000000,0.000000,0.003884,0.002251,0.003100,0.002268,0.000000,0.000000,0.000000
MIR1302-9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
MT-ND6,0.078342,0.668662,0.351375,0.488483,0.338227,0.590447,0.571153,0.549528,0.683096,0.062900,0.136474,0.070176
MT-TE,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
MT-CYB,1.869700,4.508285,4.058480,4.450702,3.867831,4.499810,4.249127,4.417264,4.692368,1.424969,1.987037,1.524643
MT-TT,0.000000,0.000000,0.000000,0.000000,0.000000,0.001582,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [3]:
geneanno_sort = pd.read_csv('/media/ggj/Guo-4T-AI/FYT/MultiOmic/CodeTest/20240819_epigen_pred_expr/geneanno_sort.csv')
order=geneanno_sort.sort_values(by='Unnamed: 0').index

Xreducedall=Xreducedall[order,:]


In [4]:
geneanno=geneanno[geneanno.symbol.isin(geneexp.index)]
geneanno

Xreducedall=Xreducedall[geneanno.index,:]
from sklearn.preprocessing import StandardScaler
Xreducedall = StandardScaler().fit_transform(Xreducedall)

geneexp=geneexp.loc[geneanno.symbol,:]
geneexp

,AST-PP,IN-PV,IN-SST,IN-SV2C,IN-VIP,L2/3,L4,L5/6,L5/6-CC,Microglia,OPC,Oligodendrocytes
TSPAN6,0.138402,0.076731,0.033782,0.043090,0.044878,0.057615,0.040370,0.032992,0.057447,0.000008,0.181110,0.002244
TNMD,0.001560,0.000000,0.000000,0.000000,0.002869,0.003998,0.004457,0.002763,0.003310,0.000000,0.002960,0.000000
DPM1,0.502191,1.126742,0.730473,1.066798,0.890244,1.506282,1.324882,1.198046,1.567789,0.219577,0.388879,0.316223
SCYL3,0.266279,0.501499,0.369091,0.491573,0.363311,0.806933,0.702744,0.625094,0.790471,0.132823,0.236863,0.155510
C1orf112,0.151163,0.504930,0.331905,0.422079,0.312763,0.764379,0.563371,0.547358,0.577204,0.395092,0.220026,0.155851
...,...,...,...,...,...,...,...,...,...,...,...,...
RP11-324D17.2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
RP11-138H10.2,0.000000,0.009461,0.000022,0.000000,0.002245,0.001080,0.002497,0.000000,0.002653,0.000000,0.000710,0.000000
RP11-327J17.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
RP11-16E12.2,0.000000,0.000000,0.001855,0.000056,0.007359,0.006002,0.006999,0.002749,0.012596,0.000000,0.000551,0.000264


In [5]:
Xreducedall.shape

(23901, 124410)

In [6]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler

In [7]:
import pickle

In [8]:
#
target_idx=range(12)
percentile_cutoff=25


tissue_list=[]
scc_list=[]
pcc_list=[]
auc_list=[]
for i in target_idx:
    print(geneexp.columns[i])
    truth_expr=geneexp
    percentile_cutoff=25
    binarized_high_truth = truth_expr > np.percentile(truth_expr, 100-percentile_cutoff, axis=0)
    binarized_low_truth = truth_expr <= np.percentile(truth_expr, percentile_cutoff, axis=0)
    on_off_genes = np.logical_or(binarized_high_truth.iloc[:,i], binarized_low_truth.iloc[:,i]).values
    
    filt = np.asarray(geneanno.iloc[:, -4] != 'rRNA')
    tmp=np.isfinite(np.asarray(
        np.log(geneexp.iloc[:, i] + 0.0001)))
    filt = filt * tmp
    trainind = np.asarray(geneanno['seqnames'] != 'chrX') * np.asarray(
geneanno['seqnames'] != 'chrY') * np.asarray(geneanno['seqnames'] != 'chr8')
    testind = np.asarray(geneanno['seqnames'].isin(['chr8']))
    xtrain = Xreducedall[trainind * filt*on_off_genes, :]
    xtest = Xreducedall[(testind) * filt*on_off_genes, :]
    ytrain=np.asarray(np.log(geneexp.iloc[trainind * filt*on_off_genes, i] + 0.0001))
    ytest=np.asarray(np.log(geneexp.iloc[(testind) * filt*on_off_genes, i] + 0.0001))
    ridge=Ridge(alpha=10)
    ridge.fit(xtrain,ytrain)
    ypred = ridge.predict(xtest)
    ytarget=np.asarray(
     np.log(geneexp.iloc[(testind) * filt*on_off_genes, i] + 0.0001))
    scc,_=spearmanr(ypred, ytarget)
    pcc,_=pearsonr(ypred, ytarget)
    
    scaler=MinMaxScaler(feature_range=(0,1))
    scaler.fit(ypred.reshape(-1, 1))
    ypred_scale=scaler.transform(ypred.reshape(-1, 1))
    ytarget_binary=np.where(geneexp.iloc[testind * filt*on_off_genes, i]>np.percentile(truth_expr, 100-percentile_cutoff, axis=0)[i],1,0)
    auc=roc_auc_score(y_true=ytarget_binary.reshape(-1,1),y_score=ypred_scale)
    
    print(scc,pcc,auc)
    print(i," :done")
    tissue_list.append(geneexp.columns[i])
    scc_list.append(scc)
    pcc_list.append(pcc)
    auc_list.append(auc)
    ct=geneexp.columns[i]
    ct=ct.replace("/", "_")
    ct=ct.replace(" ", "_")  
    with open(ct + '_ridge.pkl','wb') as f:
        pickle.dump(ridge,f)
pd.DataFrame({"Cell_type":tissue_list, "SCC":scc_list,'PCC':pcc_list,'AUC':auc_list}).T.to_csv('./Metric_magic.csv')

AST-PP
0.7634436162941854 0.8422323086529656 0.9602564102564103
0  :done
IN-PV
0.7427512921267636 0.8388303253711011 0.9661402229045446
1  :done
IN-SST
0.7532814418361734 0.834726973468368 0.9612667305712358
2  :done
IN-SV2C
0.7569422935402512 0.8350157495187753 0.9564315999501185
3  :done
IN-VIP
0.7464575388658536 0.8421865455243671 0.9644544997486174
4  :done
L2/3
0.7486374089711871 0.8389888134831357 0.9640983718236936
5  :done
L4
0.7351496167065141 0.8228052449419062 0.948581622302284
6  :done
L5/6
0.7524663637018909 0.8268652904318919 0.9562022689775995
7  :done
L5/6-CC
0.7573983093758935 0.8499115258233804 0.9691810721145451
8  :done
Microglia
0.7523415024542048 0.8014163882349645 0.9477024625539477
9  :done
OPC
0.7642887160481223 0.8500174822215051 0.9651963241436925
10  :done
Oligodendrocytes
0.7483372259538155 0.8430701650221684 0.9663343421755386
11  :done
